In [31]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import joblib
import json

# LOAD DATA

df = pd.read_csv("../../Datasets/O1D1/Ponds1.csv")

# Basic inspection
print(df.head())
print(df.info())
print(df.describe())


    station        Date      Time NITRATE(PPM)   PH  AMMONIA(mg/l)   TEMP  \
0  station1  01-02-2022  08:00:00         18.3  5.7          0.010  23.20   
1  station1  01-02-2022  08:20:00          3.6  5.1          0.094  23.41   
2  station1  01-02-2022  08:40:00         13.1  5.5          0.060  23.63   
3  station1  01-02-2022  09:00:00         18.1  5.2          0.018  23.64   
4  station1  01-02-2022  09:20:00         10.8  5.2          0.038  23.81   

     DO  TURBIDITY MANGANESE(mg/l)  
0  11.6       31.7            0.71  
1  10.5       18.8            0.62  
2  10.3       23.2            0.73  
3   9.4       26.7            0.64  
4   8.8       19.5            0.68  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74796 entries, 0 to 74795
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   station          74796 non-null  object 
 1   Date             74796 non-null  object 
 2   Time             

C:\Users\AdityaPandey\AppData\Local\Temp\ipykernel_19656\1587590797.py:9: DtypeWarning: Columns (3,4,7,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../../Datasets/O1D1/Ponds1.csv")


In [32]:
# Combine Date and Time into one datetime column
df["timestamp"] = pd.to_datetime(df["Date"] + " " + df["Time"], dayfirst=True)
df = df.sort_values("timestamp")
df = df.set_index("timestamp")
df = df.drop(columns=["Date", "Time"])


In [33]:
# Cyclical encoding
df["hour"] = df.index.hour
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df = df.drop(columns=["hour"])


In [34]:
# Convert all columns to numeric (invalid values → NaN)
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.ffill().bfill()


In [35]:
# Apply rolling mean smoothing
df_smoothed = df.rolling(window=3, min_periods=1).mean()
df_smoothed = df_smoothed.ffill().bfill()


In [36]:
# =========================
# COLUMN RENAMING
# =========================

df_smoothed.columns = [
    "station",
    "nitrate",
    "ph",
    "ammonia",
    "temperature",
    "do",
    "turbidity",
    "manganese",
    "hour_sin",
    "hour_cos"
]

# Drop station (not a feature)
df_features = df_smoothed.drop(columns=["station"])

# =========================
# INTERACTION FEATURES (BEFORE SCALING)
# =========================

df_features["ammonia_ph"] = df_features["ammonia"] * df_features["ph"]
df_features["ammonia_temp"] = df_features["ammonia"] * df_features["temperature"]

# =========================
# SCALING (FINAL STEP)
# =========================

scaler = MinMaxScaler()
scaled_values = scaler.fit_transform(df_features)

df_scaled = pd.DataFrame(
    scaled_values,
    index=df_features.index,
    columns=df_features.columns
)

# =========================
# SAVE METADATA
# =========================

columns_list = list(df_scaled.columns)

with open("../models/feature_columns.json", "w") as f:
    json.dump(columns_list, f)

joblib.dump(scaler, "../models/scaler.save")

['../models/scaler.save']

In [37]:
# Model 1: Water condition forecasting
model1_targets = ["do", "temperature", "ph", "turbidity"]

# Model 2: Nitrogen forecasting
model2_targets = ["ammonia", "nitrate"]

print(df_scaled.columns)


Index(['nitrate', 'ph', 'ammonia', 'temperature', 'do', 'turbidity',
       'manganese', 'hour_sin', 'hour_cos', 'ammonia_ph', 'ammonia_temp'],
      dtype='object')


In [38]:
import numpy as np

def create_sequences(data, target_cols, lookback=48, forecast_horizon=6):
    X, y = [], []
    
    data_values = data.values.astype(np.float32)
    target_indices = [data.columns.get_loc(col) for col in target_cols]
    
    for i in range(len(data) - lookback - forecast_horizon):
        X.append(data_values[i:i+lookback])
        y.append(data_values[i+lookback:i+lookback+forecast_horizon, target_indices])
    
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


In [39]:
# Model 1 sequences
X1, y1 = create_sequences(df_scaled, model1_targets)

# Model 2 sequences
X2, y2 = create_sequences(df_scaled, model2_targets)

print("Model 1 Input shape:", X1.shape)
print("Model 1 Output shape:", y1.shape)

print("Model 2 Input shape:", X2.shape)
print("Model 2 Output shape:", y2.shape)


Model 1 Input shape: (74742, 48, 11)
Model 1 Output shape: (74742, 6, 4)
Model 2 Input shape: (74742, 48, 11)
Model 2 Output shape: (74742, 6, 2)


In [40]:
split = int(0.8 * len(X1))

# Model 1 split
X1_train, X1_test = X1[:split], X1[split:]
y1_train, y1_test = y1[:split], y1[split:]

# Model 2 split
X2_train, X2_test = X2[:split], X2[split:]
y2_train, y2_test = y2[:split], y2[split:]


In [41]:
import numpy as np

np.save("../models/X1_train.npy", X1_train)
np.save("../models/X1_test.npy", X1_test)
np.save("../models/y1_train.npy", y1_train)
np.save("../models/y1_test.npy", y1_test)

np.save("../models/X2_train.npy", X2_train)
np.save("../models/X2_test.npy", X2_test)
np.save("../models/y2_train.npy", y2_train)
np.save("../models/y2_test.npy", y2_test)
